## TENSORFLOW

In [20]:
import tensorflow as tf
import numpy as np

## DATA PREPROCESSING AND NORMALIZING

In [21]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


### TEACHER MODEL

In [22]:
def create_teacher():
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation="relu"),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(10)
    ])


### TRAIN TEACHER MODEL

In [23]:
teacher = create_teacher()

teacher.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

teacher.fit(x_train, y_train, epochs=3, batch_size=64)

Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8990 - loss: 0.3459
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9756 - loss: 0.0800
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9846 - loss: 0.0490


### STUDENT MODEL

In [24]:
def create_student():
    return tf.keras.Sequential([
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(10)
    ])

### DISTILLATION

In [32]:
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher, temperature=2.0, alpha=0.7):
        super().__init__()
        self.teacher = teacher
        self.student = student
        self.temperature = temperature
        self.alpha = alpha

    def compile(self, optimizer):
        super().compile()
        self.optimizer = optimizer
        self.student_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        self.distillation_loss_fn = tf.keras.losses.KLDivergence()

    def call(self, inputs, training=False):
        # When model is called normally, return student predictions
        return self.student(inputs, training=training)

    def train_step(self, data):
        x, y = data

        # Teacher forward (no training)
        teacher_logits = self.teacher(x, training=False)

        with tf.GradientTape() as tape:
            student_logits = self.student(x, training=True)

            # Hard loss
            student_loss = self.student_loss_fn(y, student_logits)

            # Soft loss
            teacher_probs = tf.nn.softmax(teacher_logits / self.temperature)
            student_probs = tf.nn.softmax(student_logits / self.temperature)

            distill_loss = self.distillation_loss_fn(
                teacher_probs, student_probs
            ) * (self.temperature ** 2)

            loss = self.alpha * distill_loss + (1 - self.alpha) * student_loss

        grads = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))

        return {"loss": loss}


In [33]:
teacher = create_teacher()

teacher.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

teacher.fit(x_train, y_train, epochs=3, batch_size=64)


Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8898 - loss: 0.3607
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9755 - loss: 0.0817
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9853 - loss: 0.0457


In [34]:
teacher.trainable = False


In [35]:
student = create_student()


In [36]:
distiller = Distiller(
    student=student,
    teacher=teacher,
    temperature=2.0,
    alpha=0.7
)


In [37]:
distiller.compile(
    optimizer=tf.keras.optimizers.Adam()
)


In [38]:
distiller.fit(x_train, y_train, epochs=3, batch_size=64)


Epoch 1/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 0.7904
Epoch 2/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.2670
Epoch 3/3
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.1577


In [39]:
student.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

student.evaluate(x_test, y_test)


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9665 - loss: 0.1120


[0.09647373855113983, 0.9714000225067139]

In [40]:
student.save("distilled_student.keras")